# Session 2: From Static Weights to Adaptive Allocation — Designing the AI Rebalancing Engine

In Session 1, we built a classical minimum-variance portfolio and watched it fail under stress. The core problem? Static weights can't respond to a changing world. In this session, we take humans _partially_ out of the loop: we design an AI rebalancing engine that uses **utility maximization** to allocate capital, reads market sentiment to adjust risk preferences dynamically, and enforces explicit safety rules.

> __Learning Objectives:__
>
> * **Cobb-Douglas Utility Maximization**: Formulate portfolio allocation as a budget-constrained Cobb-Douglas utility problem, derive the analytical solution, and compare it to CES and log-linear alternatives
> * **Sentiment-Driven Preference Weights**: Construct a sentiment signal from EMA crossover and feed it into preference weights via the Single Index Model, so the allocator adapts dynamically to market conditions
> * **Rebalancing Engine with Safety Rules**: Implement a production-style rebalancing loop with trigger rules — drawdown limits, turnover caps, and de-risking protocols — that encode human safety judgment

Let's build a machine that adapts.

## Examples
The following example notebooks accompany this lecture. They contain executable code that implements the concepts discussed here:

\u27a1 [Example: Building the Cobb-Douglas Allocator](eCornell-AI-Finance-S2-Example-BuildCobbDouglasAllocator-May-2026.ipynb). In this example, we generate a synthetic market path, compute SIM parameters and the sentiment signal, build the Cobb-Douglas utility allocator, and compare its allocation behavior to CES and log-linear alternatives.

\u27a1 [Example: Rebalancing Engine and Scorecard](eCornell-AI-Finance-S2-Example-RebalancingEngineScorecard-May-2026.ipynb). In this example, we wire the Cobb-Douglas allocator into a full rebalancing engine with trigger rules, produce a scorecard comparing strategies, and run sensitivity analysis on key parameters.

## Why Static Weights Fail

Session 1 showed us that the minimum-variance optimizer treats its inputs — the expected return vector $\boldsymbol{\mu}$ and the covariance matrix $\boldsymbol{\Sigma}$ — as fixed constants. But markets are not stationary. Correlations spike during crises, volatilities cluster, and expected returns shift with economic regimes. A portfolio computed from last year's data has no mechanism to react when the world changes _today_.

> **The core limitation.** Static portfolio construction is a _one-shot_ decision: estimate, optimize, deploy. There is no feedback loop. No matter what happens after deployment — a war, a rate hike, a pandemic — the weights don't move until a human intervenes. This is not a flaw in the math; it's a flaw in the _architecture_.

The fix is straightforward in concept: replace the one-shot decision with a _loop_ that continuously reads market conditions and adjusts the portfolio accordingly. But we need a principled framework for making those allocation decisions. That framework is **utility maximization**.

## Utility Maximization: A Framework for Allocation

Instead of asking "what portfolio minimizes variance?" we ask a different question: **given a budget and current market conditions, what portfolio maximizes the investor's utility?**

This shift is fundamental. Minimum-variance optimization treats all assets symmetrically — the only inputs are returns and covariances. Utility maximization lets us encode _preferences_: which assets do we like right now? How strongly? And those preferences can change every day based on market signals.

> **The general problem.** Given $K$ assets with current prices $P_1, \ldots, P_K$ and a total budget $B$, choose the number of shares $n_1, \ldots, n_K$ to:
>
> $$\max_{n_1, \ldots, n_K} \quad U(n_1, \ldots, n_K) \qquad \text{subject to} \quad \sum_{i=1}^{K} n_i \cdot P_i = B$$
>
> The utility function $U$ encodes the investor's preferences. Different choices of $U$ lead to different allocation strategies.

The key insight is that the preference weights $\gamma_i$ inside $U$ are not fixed — they are computed _dynamically_ from market sentiment and fundamentals. This is where the AI enters: it translates real-time signals into preference weights that drive the allocation.

## Cobb-Douglas Utility

The **Cobb-Douglas utility function** is the workhorse of this course. It has a clean analytical solution, encodes preferences naturally through exponents, and separates assets into "preferred" and "non-preferred" categories.

> **Cobb-Douglas Utility.**
>
> $$U(n_1, \ldots, n_K) = \kappa(\boldsymbol{\gamma}) \prod_{i=1}^{K} n_i^{\gamma_i}$$
>
> where $\gamma_i \in (-1, 1)$ is the preference exponent for asset $i$, and $\kappa = +1$ if all $\gamma_i \geq 0$, else $\kappa = -1$.

The preference exponents do all the work:
* $\gamma_i > 0$ — **preferred**: the investor wants more of this asset. Higher $\gamma_i$ means stronger preference.
* $\gamma_i \leq 0$ — **non-preferred**: the investor wants to minimize exposure. These assets receive only a minimum position $\epsilon$.

### Analytical Solution

The budget-constrained Cobb-Douglas problem has a closed-form solution (no numerical optimizer needed).

Partition assets into preferred $S^+ = \{i : \gamma_i > 0\}$ and non-preferred $S^- = \{i : \gamma_i \leq 0\}$. Then:

> **Optimal allocation.**
>
> $$n_i^{\star} = \frac{\gamma_i}{\sum_{j \in S^+} \gamma_j} \cdot \frac{B_{\text{adj}}}{P_i} \qquad \text{for } i \in S^+$$
>
> $$n_i^{\star} = \epsilon \qquad \text{for } i \in S^-$$
>
> where $B_{\text{adj}} = B - \epsilon \sum_{k \in S^-} P_k$ is the budget remaining after funding minimum positions.

This is elegant: the budget is split among preferred assets in proportion to their preference weights, adjusted for price. The allocation responds immediately to changes in $\gamma_i$ — no matrix inversion, no quadratic program, no numerical solver.

## The Sentiment Signal: EMA Crossover → $\lambda$

The Cobb-Douglas allocator needs preference weights $\gamma_i$ as input. Where do they come from? From a combination of **fundamental signals** (the Single Index Model) and **market sentiment** (EMA crossover). The sentiment signal $\lambda_t$ acts as a risk-aversion dial that modulates how aggressively the allocator favors high-beta assets.

> **Exponential Moving Average.** The EMA of a price series $S_t$ with window $N$ is:
>
> $$\bar{S}_{t} = \alpha\, S_{t} + (1 - \alpha)\,\bar{S}_{t-1}, \qquad \alpha = \frac{2}{N + 1}$$
>
> We compute two EMAs: a **short-term** ($N_0 = 21$ days, ~1 month) that tracks recent momentum, and a **long-term** ($N_1 = 63$ days, ~1 quarter) that tracks the underlying trend.

The crossover between short and long EMAs produces the sentiment signal:

> **The Lambda Signal.**
>
> $$\lambda_t = -G \cdot \left(\frac{\bar{S}^{\text{short}}_t}{\bar{S}^{\text{long}}_t} - 1\right)$$
>
> where $G > 0$ is a gain constant that controls sensitivity. The interpretation:
> * $\lambda_t > 0$ \u2192 **Bearish** (short EMA below long EMA). The engine becomes risk-averse.
> * $\lambda_t < 0$ \u2192 **Bullish** (short EMA above long EMA). The engine takes on more risk.
> * $\lambda_t \approx 0$ \u2192 **Neutral** (no strong signal).

## From Sentiment to Preferences: The SIM-Based $\gamma_i$

Each asset $i$ has **Single Index Model (SIM) parameters** $(\alpha_i, \beta_i, \sigma_i)$ estimated from historical data:
* $\alpha_i$ — firm-specific excess return (Jensen's alpha)
* $\beta_i$ — sensitivity to the market factor (systematic risk)
* $\sigma_i$ — residual volatility

The preference weight $\gamma_i$ combines SIM fundamentals with the sentiment signal $\lambda_t$:

> **Preference Weight Computation.**
>
> $$\gamma_i = \tanh\left(\frac{\alpha_i}{\beta_i^{\lambda_t}} + \frac{\beta_i}{\beta_i^{\lambda_t}} \cdot \mathbb{E}[g_m]\right)$$
>
> which simplifies to:
>
> $$\gamma_i = \tanh\left(\frac{\alpha_i}{\beta_i^{\lambda_t}} + \beta_i^{1-\lambda_t} \cdot \mathbb{E}[g_m]\right)$$

The risk factor $\beta_i^{\lambda_t}$ is the key mechanism:
* **When bearish** ($\lambda_t > 0$): High-beta assets are penalized exponentially — the denominator grows, pushing $\gamma_i$ toward zero or negative.
* **When bullish** ($\lambda_t < 0$): High-beta assets are amplified — the risk factor shrinks, letting the growth signal dominate.
* **When neutral** ($\lambda_t = 0$): $\beta_i^0 = 1$, so the raw SIM growth score drives preferences.

The $\tanh$ activation maps the result to $(-1, 1)$, giving us well-behaved Cobb-Douglas exponents. These preference weights then flow directly into the Cobb-Douglas analytical solution to determine share allocations.

## Alternative Utility Functions

Cobb-Douglas is not the only choice. Different utility functions encode different assumptions about how investors trade off between assets. Here we introduce two alternatives that we'll compare in the examples.

### CES (Constant Elasticity of Substitution)

> $$U_{\text{CES}}(\mathbf{n}) = \left(\sum_{i \in S^+} \gamma_i \cdot n_i^{\rho}\right)^{1/\rho}, \qquad \rho = \frac{\sigma - 1}{\sigma}$$
>
> where $\sigma > 0$ is the **elasticity of substitution** — how willing the investor is to shift allocation between assets.

CES is a generalization of Cobb-Douglas:
* As $\sigma \to 1$: CES converges to Cobb-Douglas (unit elasticity)
* As $\sigma \to \infty$: CES converges to linear utility (perfect substitutes — all-in on the best asset)
* As $\sigma \to 0$: CES converges to Leontief (perfect complements — equal allocation)

**When to use CES:** When you want to control how concentrated the allocation becomes. High $\sigma$ produces more concentrated portfolios; low $\sigma$ produces more diversified ones.

### Log-Linear Utility

> $$U_{\text{log}}(\mathbf{n}) = \sum_{i \in S^+} \gamma_i \cdot \log(n_i)$$

Log-linear utility is the logarithmic transform of Cobb-Douglas. It produces the **same optimal allocation** (the log transform preserves the optimum), but the utility _values_ differ. This matters when utility is used as a reward signal — for example, in the bandit algorithms we'll introduce in Session 3.

**When to use log-linear:** When you want the same allocation as Cobb-Douglas but need utility values that are additive (useful for averaging across scenarios or feeding into learning algorithms).

## The Rebalancing Engine: Pseudo-code

The utility allocator gives us a principled way to compute positions at any point in time. The rebalancing engine wraps this allocator in a daily loop with safety rules:

> **Phase 1: Initialization**
> 1. Generate (or load) synthetic market price path $S_t$ for a broad index.
> 2. Compute short-term EMA ($N_0 = 21$) and long-term EMA ($N_1 = 63$). Define warmup offset $t_0 = N_0 + N_1$.
> 3. Build the sentiment series: $\lambda_t = -G \cdot (\bar{S}^{\text{short}}_t / \bar{S}^{\text{long}}_t - 1)$.
> 4. Smooth market growth into $g_{m,t}$ using an EMA of log returns ($N_2 = 10$).
> 5. Assemble the **context model**: budget $B$, ticker universe, price matrix, SIM parameters, risk floor $\epsilon$, and market factor $g_{m,t}$.
> 6. Define **trigger rules**: drawdown limit, turnover cap, and reallocation schedule.
> 7. Compute initial allocation via Cobb-Douglas utility maximization.

> **Phase 2: Daily Loop** (for $t = t_0 + 1, \ldots, t_0 + T$)
> 1. Read reallocation flag $a_t$ from schedule.
> 2. **If $a_t = 1$ (rebalance day):**
>    - Liquidate current positions at today's prices
>    - **Check drawdown trigger:** if drawdown exceeds $d_{\max}$, de-risk to 100% cash
>    - Otherwise, update $\lambda_t$, compute new $\gamma_i$ preferences, solve Cobb-Douglas allocation
>    - **Check turnover cap:** if proposed trades exceed $\tau_{\max}$, scale down proportionally
> 3. **If $a_t = 0$ (hold day):** Propagate prior allocation unchanged.

> **Phase 3: Output**
> Return history indexed by trading day: positions, preference weights, cash, and total wealth.

## Trigger Rules: The Safety Net

An adaptive engine that trades freely is dangerous — it can churn the portfolio, chase noise, or fail to protect capital during a crash. Trigger rules are the _guardrails_ that keep the engine safe:

| Rule | Parameter | What It Does |
|------|-----------|-------------|
| **Drawdown Limit** | $d_{\max}$ (e.g., 10%) | If portfolio wealth drops more than $d_{\max}$ from its peak, the engine de-risks to 100% cash. This is a circuit breaker. |
| **Turnover Cap** | $\tau_{\max}$ (e.g., 50%) | If the proposed rebalance would trade more than $\tau_{\max}$ of portfolio value, the trades are scaled down proportionally. This controls transaction costs. |
| **Reallocation Schedule** | $a_t \in \{0, 1\}$ | The binary schedule determines _which days_ the engine is allowed to rebalance. Daily, weekly, or event-driven. |

> **Why are trigger rules essential?** Without them, the engine is unconstrained — it could rebalance every day (expensive), ignore a crash (catastrophic), or flip positions wildly based on noise in the lambda signal. Trigger rules encode _human judgment_ about acceptable risk into the machine's decision loop. They are the bridge between autonomous operation and investment committee oversight.

In Session 4, we'll extend these rules with human override protocols and escalation procedures for production deployment.

## Summary

> __Key Takeaways:__
>
> * **Utility maximization replaces variance minimization**: The Cobb-Douglas utility function has a clean analytical solution — allocate budget proportionally to preference weights $\gamma_i$ — and CES/log-linear alternatives offer different concentration/diversification trade-offs using the same weights
> * **Preferences adapt dynamically**: Preference weights are computed from SIM fundamentals ($\alpha_i$, $\beta_i$) modulated by a sentiment signal $\lambda_t$ from EMA crossover, so the allocator shifts risk appetite as market conditions change
> * **Trigger rules encode human judgment**: Drawdown limits, turnover caps, and reallocation schedules act as safety guardrails, ensuring the engine operates within bounds that a human investment committee would approve

**Next Session:** In [Session 3](../session-3/eCornell-AI-Finance-S3-Lecture-HMMBacktesting-May-2026.ipynb), we'll stress-test this rebalancing engine against Hidden Markov Model-generated synthetic market paths that include regime shifts. We'll also introduce **multi-armed bandits** as an adaptive challenger — can a learning algorithm beat the utility-based allocator when markets change regime?

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The rebalancing engine described here is a pedagogical tool using synthetic data and simplified models. Real-world algorithmic trading involves regulatory requirements, execution risk, and operational considerations beyond the scope of this session.